### Load cleaned data and label mapping

In [1]:
import pandas as pd
import numpy as np
import torch

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)


/Users/josephakumatey/airline-nlp-env/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
/Users/josephakumatey/airline-nlp-env/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
Twitter_nlp = pd.read_csv('Twitter_sentiment.csv')
Twitter_nlp.head()

,tweet_id,airline_sentiment,airline_sentiment_confidence,negativereason,negativereason_confidence,airline,airline_sentiment_gold,name,negativereason_gold,retweet_count,text,tweet_coord,tweet_created,tweet_location,user_timezone
0,570306133677760513,neutral,1.0000,NaN,NaN,Virgin America,NaN,cairdin,NaN,0,@VirginAmerica What @dhepburn said.,NaN,2015-02-24 11:35:52 -0800,NaN,Eastern Time (US & Canada)
1,570301130888122368,positive,0.3486,NaN,0.0000,Virgin America,NaN,jnardino,NaN,0,@VirginAmerica plus you've added commercials t...,NaN,2015-02-24 11:15:59 -0800,NaN,Pacific Time (US & Canada)
2,570301083672813571,neutral,0.6837,NaN,NaN,Virgin America,NaN,yvonnalynn,NaN,0,@VirginAmerica I didn't today... Must mean I n...,NaN,2015-02-24 11:15:48 -0800,Lets Play,Central Time (US & Canada)
3,570301031407624196,negative,1.0000,Bad Flight,0.7033,Virgin America,NaN,jnardino,NaN,0,@VirginAmerica it's really aggressive to blast...,NaN,2015-02-24 11:15:36 -0800,NaN,Pacific Time (US & Canada)
4,570300817074462722,negative,1.0000,Can't Tell,1.0000,Virgin America,NaN,jnardino,NaN,0,@VirginAmerica and it's a really big bad thing...,NaN,2015-02-24 11:14:45 -0800,NaN,Pacific Time (US & Canada)


In [3]:
Twitter_nlp = Twitter_nlp[['text', 'airline_sentiment']].dropna()

### Train/validation/test split

In [4]:
label2id = {'negative': 0, 'neutral': 1, 'positive': 2}
id2label = {v: k for k, v in label2id.items()}

Twitter_nlp['label'] = Twitter_nlp['airline_sentiment'].map(label2id)


In [5]:
X = Twitter_nlp['text'].values
y = Twitter_nlp['label'].values

X_train_val, X_test, y_train_val, y_test = train_test_split(
    X, y, test_size=0.1, random_state=42, stratify=y
)

X_train, X_val, y_train, y_val = train_test_split(
    X_train_val, y_train_val, test_size=0.1111, random_state=42, stratify=y_train_val
)


### Tokenization with DistilBERT tokenizer

In [6]:
model_name = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize_batch(texts):
    return tokenizer(
        list(texts),
        truncation=True,
        padding="max_length",
        max_length=64
    )


In [7]:
train_encodings = tokenize_batch(X_train)
val_encodings   = tokenize_batch(X_val)
test_encodings  = tokenize_batch(X_test)


### Dataset and Dataloader / HF Dataset creation

In [8]:
class AirlineDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {k: torch.tensor(v[idx]) for k, v in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx])
        return item

train_dataset = AirlineDataset(train_encodings, y_train)
val_dataset   = AirlineDataset(val_encodings, y_val)
test_dataset  = AirlineDataset(test_encodings, y_test)


### Model setup (DistilBERTForSequenceClassification)

In [9]:
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=3,
    id2label=id2label,
    label2id=label2id
)


Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


### Training loop / Trainer config

In [10]:
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)


In [11]:
training_args = TrainingArguments(
    output_dir="./models/distilbert",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=3,
    weight_decay=0.01,
    logging_steps=50,
)


### Evaluation on test set (metrics + confusion matrix)

In [12]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = logits.argmax(axis=-1)
    acc = accuracy_score(labels, preds)
    report = classification_report(labels, preds, output_dict=True, zero_division=0)
    macro_f1 = report["macro avg"]["f1-score"]
    return {
        "accuracy": acc,
        "macro_f1": macro_f1
    }

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
)


/var/folders/_1/_cwdv6pj4b51jm3yc1c1jy340000gn/T/ipykernel_16096/3154260784.py:12: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


In [13]:
trainer.train()


Step,Training Loss
50,0.828000
100,0.696300
150,0.555300
200,0.545600
250,0.502300
300,0.463200
350,0.468000
400,0.445600
450,0.487500
500,0.454100


TrainOutput(global_step=2166, training_loss=0.3623518952592478, metrics={'train_runtime': 17246.3667, 'train_samples_per_second': 2.008, 'train_steps_per_second': 0.126, 'total_flos': 573312566991744.0, 'train_loss': 0.3623518952592478, 'epoch': 3.0})

### Compare with baseline metrics

In [14]:
preds_output = trainer.predict(test_dataset)
logits = preds_output.predictions
y_pred = logits.argmax(axis=-1)

print("Test accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(
    y_test,
    y_pred,
    target_names=['negative','neutral','positive'],
    zero_division=0
))



Test accuracy: 0.8440748440748441
              precision    recall  f1-score   support

    negative       0.89      0.93      0.91       908
     neutral       0.73      0.65      0.69       306
    positive       0.77      0.76      0.77       229

    accuracy                           0.84      1443
   macro avg       0.80      0.78      0.79      1443
weighted avg       0.84      0.84      0.84      1443



### Example tweets and prediction

In [25]:
from transformers import AutoModelForSequenceClassification, AutoTokenizer

model_path = "./models/distilbert-best"  # match whatever you used above

tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForSequenceClassification.from_pretrained(model_path)
model.to(device)
model.eval()


DistilBertForSequenceClassification(
  (distilbert): DistilBertModel(
    (embeddings): Embeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (transformer): Transformer(
      (layer): ModuleList(
        (0-5): 6 x TransformerBlock(
          (attention): DistilBertSdpaAttention(
            (dropout): Dropout(p=0.1, inplace=False)
            (q_lin): Linear(in_features=768, out_features=768, bias=True)
            (k_lin): Linear(in_features=768, out_features=768, bias=True)
            (v_lin): Linear(in_features=768, out_features=768, bias=True)
            (out_lin): Linear(in_features=768, out_features=768, bias=True)
          )
          (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
          (ffn): FFN(
            (dropout): Dropout(p=0.1, inplace=False)


In [26]:
label2id = {"negative": 0, "neutral": 1, "positive": 2}
id2label = {v: k for k, v in label2id.items()}

example_tweets = [
    "My flight was delayed for 5 hours and nobody helped at the gate.",
    "Thanks @Delta for the smooth check-in and super friendly staff!",
    "Hi @United, what is the baggage allowance for international flights?",
    "Plane is fine but the customer service on the phone was terrible.",
    "Boarded early, took off on time, and landed early. Great job!",
]

for text in example_tweets:
    inputs = tokenizer(
        text,
        truncation=True,
        padding=True,
        return_tensors="pt"
    ).to(device)

    with torch.no_grad():
        outputs = model(**inputs)

    logits = outputs.logits
    pred_id = int(logits.argmax(dim=-1)[0])
    pred_label = id2label[pred_id]

    print(f"Tweet: {text}")
    print(f"Model prediction: {pred_label}")
    print("-" * 80)


Tweet: My flight was delayed for 5 hours and nobody helped at the gate.
Model prediction: negative
--------------------------------------------------------------------------------
Tweet: Thanks @Delta for the smooth check-in and super friendly staff!
Model prediction: positive
--------------------------------------------------------------------------------
Tweet: Hi @United, what is the baggage allowance for international flights?
Model prediction: neutral
--------------------------------------------------------------------------------
Tweet: Plane is fine but the customer service on the phone was terrible.
Model prediction: negative
--------------------------------------------------------------------------------
Tweet: Boarded early, took off on time, and landed early. Great job!
Model prediction: positive
--------------------------------------------------------------------------------


### Error analysis and final discussion

#### Error analysis

To better understand the remaining weaknesses of the DistilBERT model, I inspected misclassified test examples by comparing the true labels with the model’s predictions. Several patterns emerged:



Neutral vs negative confusion: Many errors occur where tweets are labeled as neutral but contain mild frustration or sarcasm, or where the tweet mixes a complaint with a neutral factual statement. These cases are ambiguous even for humans, and the model often predicts negative instead of neutral.

Neutral vs positive confusion: Some tweets labeled neutral contain appreciative phrases (“thanks”, “great”) embedded in informational messages, leading the model to predict positive sentiment. This suggests the model strongly relies on sentiment‑bearing keywords even when the overall tweet is intended to be neutral.

Short / context‑poor tweets: Very short tweets (few words, abbreviations, or just a brand mention) are harder to classify reliably because they provide limited context. The model sometimes defaults to negative for short, vague messages, likely influenced by the overall class imbalance toward negative tweets in the dataset.



In [16]:
import pandas as pd

# y_test: true numeric labels (0=neg,1=neu,2=pos)
# y_pred: predicted numeric labels from DistilBERT
# X_test: original tweet texts

error_indices = np.where(y_test != y_pred)[0]

# Build a small DataFrame of misclassified examples
errors_df = pd.DataFrame({
    "text": X_test[error_indices],
    "true_label": [id2label[int(l)] for l in y_test[error_indices]],
    "pred_label": [id2label[int(l)] for l in y_pred[error_indices]]
})

errors_df.head(10)


,text,true_label,pred_label
0,@AmericanAir Just followed you.,positive,neutral
1,@united I forgot that Intl flights out of LAX ...,positive,negative
2,@JetBlue yes. They are working on it. Hoping b...,negative,neutral
3,"@SouthwestAir For my Grandma Ella's 80th, she ...",neutral,positive
4,@united Flt 359 lax to EWR. Your pilot bragged...,negative,positive
5,@USAirways thx 4 replying. After trying 2 get ...,positive,negative
6,@united she was at the service desk at gate 21...,negative,neutral
7,"@jetblue captain ""takes as lot of muscles to f...",neutral,positive
8,@AmericanAir @cityandsand we're trying to coor...,neutral,negative
9,@united received my bag. I appreciate taking c...,positive,negative


#### Final discussion

Overall, the fine‑tuned DistilBERT model significantly improves performance over the TF–IDF baselines, especially for neutral and positive classes, reaching approximately 84% test accuracy and 0.79 macro F1 on the 3‑class sentiment task. However, remaining errors highlight the intrinsic ambiguity of short social‑media text and the difficulty of distinguishing neutral from weakly positive or negative statements. Future improvements could include experimenting with larger transformer models, using class‑balanced or focal loss to further address minority classes, incorporating additional context (e.g., conversation threads or user history), or reframing the task as binary sentiment (negative vs non‑negative) for operational use cases where the primary goal is to catch complain